Cell 1: Git Clone or Update

In [1]:
import os
REPO_URL = "https://github.com/Maryam-Faatima/clinical_trial_matching_llm.git"
REPO_DIR = "/kaggle/working/clinical_trial_matching_llm"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!ls

Cloning into '/kaggle/working/clinical_trial_matching_llm'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 140 (delta 52), reused 121 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.04 MiB | 8.61 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/kaggle/working/clinical_trial_matching_llm
app.py	data  logs  outputs  README.md	requirements.txt  results  src


Install Dependencies

In [3]:
!pip install -q -r requirements.txt

Check GPU

In [4]:
!nvidia-smi
import torch
print(torch.__version__, "| CUDA available:", torch.cuda.is_available())

Sun Nov  8 13:37:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Stage 1**

In [11]:
!python -m src.data_prep
!python -m src.model_info
!python -m src.token_stats

raw columns: ['annotation_id', 'patient_id', 'note', 'trial_id', 'trial_title', 'criterion_type', 'criterion_text', 'gpt4_explanation', 'explanation_correctness', 'gpt4_sentences', 'expert_sentences', 'gpt4_eligibility', 'expert_eligibility', 'training'] | rows: 1015
null counts:
 annotation_id         0
patient_id            0
note                  0
trial_id              0
trial_title           0
criterion_type        0
criterion_text        1
expert_sentences      0
expert_eligibility    0
dtype: int64
excluded rows by reason:
 reason
missing_criterion_text    1
Name: count, dtype: int64
expert_eligibility  excluded  included  not applicable  not enough information  not excluded  not included
criterion_type                                                                                            
exclusion                 10         0              46                     119           483             0
inclusion                  0       150               9                     174   

In [12]:
!ls -la data data/processed
!head -c 600 data/split_manifest.json

data:
total 24
drwxr-xr-x 3 root root 4096 Nov  8 09:40 .
drwxr-xr-x 5 root root 4096 Nov  8 09:31 ..
-rw-r--r-- 1 root root  130 Nov  8 09:40 excluded_rows.csv
-rw-r--r-- 1 root root    0 Nov  8 09:31 .gitkeep
drwxr-xr-x 2 root root 4096 Nov  8 09:33 processed
-rw-r--r-- 1 root root 5098 Nov  8 09:40 split_manifest.json

data/processed:
total 964
drwxr-xr-x 2 root root   4096 Nov  8 09:33 .
drwxr-xr-x 3 root root   4096 Nov  8 09:40 ..
-rw-r--r-- 1 root root      0 Nov  8 09:31 .gitkeep
-rw-r--r-- 1 root root 181389 Nov  8 09:40 test.jsonl
-rw-r--r-- 1 root root 579429 Nov  8 09:40 train.jsonl
-rw-r--r-- 1 root root 210494 Nov  8 09:40 val.jsonl
{
  "dataset": "ncbi/TrialGPT-Criterion-Annotations",
  "seed": 42,
  "raw_rows": 1015,
  "method": "sorted unique patient IDs -> np.random.default_rng(42).shuffle -> first 37 / next 8 / last 8 (computed on all raw patients)",
  "excluded_rows_total": 1,
  "excluded_by_reason": {
    "missing_criterion_text": 1
  },
  "splits": {
    "train": 

**Stage 2**

In [14]:
!pip uninstall -y torchao
!python -m src.train_lora

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 249.70it/s]
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/clinical_trial_matching_llm/src/train_lora.py", line 141, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/clinical_trial_matching_llm/src/train_lora.py", line 106, in main
    scaler.scale(loss).backward()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        self, gradient, retain_graph, create_graph, inputs=inputs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/

In [13]:
!git pull
!python -m src.evaluate --split val --name base_val          # "before" (zero-shot)
!python -m src.train_lora                                     # fine-tune, ~3 epochs
!python -m src.evaluate --split val --adapter outputs/lora_r16/epoch_1 --name ft_e1_val
!python -m src.evaluate --split val --adapter outputs/lora_r16/epoch_2 --name ft_e2_val
!python -m src.evaluate --split val --adapter outputs/lora_r16/epoch_3 --name ft_e3_val

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7 (delta 2), reused 7 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 6.54 KiB | 3.27 MiB/s, done.
From https://github.com/Maryam-Faatima/clinical_trial_matching_llm
   f76fdbd..88f4a9f  main       -> origin/main
Updating f76fdbd..88f4a9f
Fast-forward
 .gitignore        |   4 ++
 src/evaluate.py   |  85 ++++++++++++++++++++++++++++++++
 src/metrics.py    | 120 ++++++++++++++++++++++++++++++++++++++++++++++
 src/train_lora.py | 141 ++++++++++++++++++++++++++++++++++++++++++++++++++++++
 4 files changed, 350 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 src/evaluate.py
 create mode 100644 src/metrics.py
 create mode 100644 src/train_lora.py
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 250.60it/s]
generated 206 outputs in 30.0s

=== OVERALL (n=206, invalid=0) ===
accuracy=0.107  macro_f1=0

In [15]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python -m src.train_lora

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 239.24it/s]
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
epoch 1 step 10/114 loss 0.1027
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/clinical_trial_matching_llm/src/train_lora.py", line 141, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/clinical_trial_matching_llm/src/train_lora.py", line 105, in main
    loss = model(**batch).loss / CFG["grad_accum"]
           ~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args,

In [18]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python -m src.train_lora

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 256.60it/s]
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
epoch 1 step 10/114 loss 0.1036
epoch 1 step 20/114 loss 0.1670
epoch 1 step 30/114 loss 0.1544
== epoch 1 val_loss 0.1327  (277s)
epoch 2 step 40/114 loss 0.0953
epoch 2 step 50/114 loss 0.0661
epoch 2 step 60/114 loss 0.1034
epoch 2 step 70/114 loss 0.1115
== epoch 2 val_loss 0.1255  (556s)
epoch 3 step 80/114 loss 0.0688
epoch 3 step 90/114 loss 0.0796
epoch 3 step 100/114 loss 0.0252
epoch 3 step 110/114 loss 0.0729
== epoch 3 val_loss 0.1300  (834s)
peak GPU mem GiB: 7.94559383392334


In [19]:
!zip -r results_stage2.zip outputs data/split_manifest.json data/excluded_rows.csv

  adding: outputs/ (stored 0%)
  adding: outputs/lora_r16/ (stored 0%)
  adding: outputs/lora_r16/train_log.json (deflated 78%)
  adding: outputs/lora_r16/chat_template.jinja (deflated 71%)
  adding: outputs/lora_r16/tokenizer.json (deflated 81%)
  adding: outputs/lora_r16/epoch_1/ (stored 0%)
  adding: outputs/lora_r16/epoch_1/adapter_config.json (deflated 61%)
  adding: outputs/lora_r16/epoch_1/adapter_model.safetensors (deflated 8%)
  adding: outputs/lora_r16/epoch_1/README.md (deflated 65%)
  adding: outputs/lora_r16/epoch_2/ (stored 0%)
  adding: outputs/lora_r16/epoch_2/adapter_config.json (deflated 61%)
  adding: outputs/lora_r16/epoch_2/adapter_model.safetensors (deflated 8%)
  adding: outputs/lora_r16/epoch_2/README.md (deflated 65%)
  adding: outputs/lora_r16/tokenizer_config.json (deflated 59%)
  adding: outputs/lora_r16/epoch_3/ (stored 0%)
  adding: outputs/lora_r16/epoch_3/adapter_config.json (deflated 61%)
  adding: outputs/lora_r16/epoch_3/adapter_model.safetensors (def

In [20]:
!python -m src.evaluate --split val --adapter outputs/lora_r16/epoch_1 --name ft_e1_val
!python -m src.evaluate --split val --adapter outputs/lora_r16/epoch_2 --name ft_e2_val
!python -m src.evaluate --split val --adapter outputs/lora_r16/epoch_3 --name ft_e3_val

Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 253.42it/s]
generated 206 outputs in 21.4s

=== OVERALL (n=206, invalid=0) ===
accuracy=0.728  macro_f1=0.353  macro_f1(gold classes)=0.353
  included                 P=0.750 R=0.857 F1=0.800 support=35
  not included             P=0.000 R=0.000 F1=0.000 support=3
  not enough information   P=0.731 R=0.358 F1=0.481 support=53
  not applicable           P=0.000 R=0.000 F1=0.000 support=12
  excluded                 P=0.000 R=0.000 F1=0.000 support=2
  not excluded             P=0.721 R=1.000 F1=0.838 support=101
  confusion (rows=gold, cols=pred): ['included', 'not included', 'not enough information', 'not applicable', 'excluded', 'not excluded']
    [30, 0, 5, 0, 0, 0]
    [2, 0, 1, 0, 0, 0]
    [8, 0, 19, 0, 0, 26]
    [0, 0, 1, 0, 0, 11]
    [0, 0, 0, 0, 0, 2]
    [0, 0, 0, 0, 0, 101]
  evidence micro P/R/F1 = 0.390/0.557/0.459  macro(nonempty gold) F1=0.521  macro(all rows) F1=0.733  abstain_ok=0.896551724137931 fa

In [24]:
!zip -r results_stage2_eval.zip outputs

updating: outputs/ (stored 0%)
updating: outputs/lora_r16/ (stored 0%)
updating: outputs/lora_r16/train_log.json (deflated 78%)
updating: outputs/lora_r16/chat_template.jinja (deflated 71%)
updating: outputs/lora_r16/tokenizer.json (deflated 81%)
updating: outputs/lora_r16/epoch_1/ (stored 0%)
updating: outputs/lora_r16/epoch_1/adapter_config.json (deflated 61%)
updating: outputs/lora_r16/epoch_1/adapter_model.safetensors (deflated 8%)
updating: outputs/lora_r16/epoch_1/README.md (deflated 65%)
updating: outputs/lora_r16/epoch_2/ (stored 0%)
updating: outputs/lora_r16/epoch_2/adapter_config.json (deflated 61%)
updating: outputs/lora_r16/epoch_2/adapter_model.safetensors (deflated 8%)
updating: outputs/lora_r16/epoch_2/README.md (deflated 65%)
updating: outputs/lora_r16/tokenizer_config.json (deflated 59%)
updating: outputs/lora_r16/epoch_3/ (stored 0%)
updating: outputs/lora_r16/epoch_3/adapter_config.json (deflated 61%)
updating: outputs/lora_r16/epoch_3/adapter_model.safetensors (def

**Stage 3**

In [5]:
%cd /kaggle/working/clinical_trial_matching_llm
!pip uninstall -y torchao
!pip install -q -r requirements.txt
!ls outputs/lora_r16/epoch_3

/kaggle/working/clinical_trial_matching_llm
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
ls: cannot access 'outputs/lora_r16/epoch_3': No such file or directory


In [22]:
!git pull

remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 20 (delta 6), reused 18 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (20/20), 137.45 KiB | 2.50 MiB/s, done.
From https://github.com/Maryam-Faatima/clinical_trial_matching_llm
   b2eb37c..cff4176  main       -> origin/main
Updating b2eb37c..cff4176
error: The following untracked working tree files would be overwritten by merge:
	data/processed/test.jsonl
	data/processed/train.jsonl
	data/processed/val.jsonl
	outputs/kv/benchmark.json
	outputs/kv/correctness_fp16.json
	outputs/kv/correctness_fp32.json
	outputs/kv/kv_benchmark.png
Please move or remove them before you merge.
Aborting


In [4]:
%cd /kaggle/working
!git clone https://github.com/Maryam-Faatima/clinical_trial_matching_llm.git
!mkdir -p clinical_trial_matching_llm/outputs/lora_r16
!mv outputs/lora_r16/epoch_3 clinical_trial_matching_llm/outputs/lora_r16/
%cd /kaggle/working/clinical_trial_matching_llm
!pip uninstall -y torchao
!pip install -q -r requirements.txt matplotlib
!ls outputs/lora_r16/epoch_3
!ls data/processed
!ls src

/kaggle/working
Cloning into 'clinical_trial_matching_llm'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 60 (delta 23), reused 52 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 99.75 KiB | 1.47 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/kaggle/working/clinical_trial_matching_llm
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
adapter_config.json  adapter_model.safetensors	README.md
bench_kv.py   __init__.py   model_info.py  token_stats.py
data_prep.py  kv_decode.py  prompts.py	   train_lora.py
evaluate.py   metrics.py    test_kv.py


In [6]:
%cd /kaggle/working/clinical_trial_matching_llm
!python -m src.data_prep
!ls data/processed

/kaggle/working/clinical_trial_matching_llm
README.md: 100%|███████████████████████████████| 893/893 [00:00<00:00, 2.64MB/s]

data/train-00000-of-00001.parquet: downloading bytes:       |  0.00B            
data/train-00000-of-00001.parquet: downloading bytes: ██████|  157kB, 15.6kB/s  
data/train-00000-of-00001.parquet: reconstructing file: 100%|█|  176kB /  176kB,
Generating train split: 100%|█████| 1015/1015 [00:00<00:00, 11360.94 examples/s]
raw columns: ['annotation_id', 'patient_id', 'note', 'trial_id', 'trial_title', 'criterion_type', 'criterion_text', 'gpt4_explanation', 'explanation_correctness', 'gpt4_sentences', 'expert_sentences', 'gpt4_eligibility', 'expert_eligibility', 'training'] | rows: 1015
null counts:
 annotation_id         0
patient_id            0
note                  0
trial_id              0
trial_title           0
criterion_type        0
criterion_text        1
expert_sentences      0
expert_eligibility    0
dtype: int64
excluded rows by reason:
 reason
missin

In [7]:
!python -m src.test_kv

Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 243.47it/s]

--- synthetic inputs, fixed length (no EOS stop), fp16 ---
{'equal': True, 'n_tokens': 32, 'first_mismatch': None, 'max_abs_logit_diff': 0.0556640625, 'mean_abs_logit_diff': 0.005488411523401737, 'argmax_agreement': 1.0, 'min_top2_gap_over_steps': 0.09375, 'input_len': 64, 'new_tokens': 32}
{'equal': True, 'n_tokens': 64, 'first_mismatch': None, 'max_abs_logit_diff': 0.0546875, 'mean_abs_logit_diff': 0.004844235721975565, 'argmax_agreement': 1.0, 'min_top2_gap_over_steps': 0.015625, 'input_len': 256, 'new_tokens': 64}
{'equal': True, 'n_tokens': 128, 'first_mismatch': None, 'max_abs_logit_diff': 0.0546875, 'mean_abs_logit_diff': 0.004878553561866283, 'argmax_agreement': 1.0, 'min_top2_gap_over_steps': 0.015625, 'input_len': 512, 'new_tokens': 128}
{'equal': True, 'n_tokens': 128, 'first_mismatch': None, 'max_abs_logit_diff': 0.0400390625, 'mean_abs_logit_diff': 0.004426996223628521, 'argmax_agreement': 1.

In [8]:
!python -m src.test_kv --dtype fp32

Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 207.26it/s]

--- synthetic inputs, fixed length (no EOS stop), fp32 ---
{'equal': True, 'n_tokens': 32, 'first_mismatch': None, 'max_abs_logit_diff': 8.20159912109375e-05, 'mean_abs_logit_diff': 5.5069021982490085e-06, 'argmax_agreement': 1.0, 'min_top2_gap_over_steps': 0.08938217163085938, 'input_len': 64, 'new_tokens': 32}
{'equal': True, 'n_tokens': 64, 'first_mismatch': None, 'max_abs_logit_diff': 0.00016999244689941406, 'mean_abs_logit_diff': 4.9810960263130255e-06, 'argmax_agreement': 1.0, 'min_top2_gap_over_steps': 0.0001506805419921875, 'input_len': 256, 'new_tokens': 64}
{'equal': True, 'n_tokens': 128, 'first_mismatch': None, 'max_abs_logit_diff': 0.0002200603485107422, 'mean_abs_logit_diff': 8.532347237633076e-06, 'argmax_agreement': 1.0, 'min_top2_gap_over_steps': 0.011312484741210938, 'input_len': 512, 'new_tokens': 128}
{'equal': True, 'n_tokens': 128, 'first_mismatch': None, 'max_abs_logit_diff': 6.0319

In [9]:
!python -m src.bench_kv

Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 249.08it/s]
in=  128 out=  32 | uncached    22.9 tok/s | cached    25.7 tok/s | speedup 1.1x | peak MiB u/c 10/12 | equal=True
in=  128 out= 128 | uncached    20.1 tok/s | cached    25.8 tok/s | speedup 1.3x | peak MiB u/c 18/13 | equal=True
in=  512 out=  32 | uncached     6.9 tok/s | cached    24.3 tok/s | speedup 3.5x | peak MiB u/c 54/62 | equal=True
in=  512 out= 128 | uncached     6.6 tok/s | cached    25.3 tok/s | speedup 3.8x | peak MiB u/c 69/62 | equal=True
in= 1024 out=  32 | uncached     3.3 tok/s | cached    21.6 tok/s | speedup 6.6x | peak MiB u/c 161/180 | equal=True
in= 1024 out= 128 | uncached     3.1 tok/s | cached    24.7 tok/s | speedup 8.0x | peak MiB u/c 188/180 | equal=True


In [17]:
!zip -r -0 -q kv_results.zip outputs/kv

In [12]:
%cd /kaggle/working/clinical_trial_matching_llm
!python -m src.data_prep
!ls data/processed
!python -m src.test_kv

/kaggle/working/clinical_trial_matching_llm
raw columns: ['annotation_id', 'patient_id', 'note', 'trial_id', 'trial_title', 'criterion_type', 'criterion_text', 'gpt4_explanation', 'explanation_correctness', 'gpt4_sentences', 'expert_sentences', 'gpt4_eligibility', 'expert_eligibility', 'training'] | rows: 1015
null counts:
 annotation_id         0
patient_id            0
note                  0
trial_id              0
trial_title           0
criterion_type        0
criterion_text        1
expert_sentences      0
expert_eligibility    0
dtype: int64
excluded rows by reason:
 reason
missing_criterion_text    1
Name: count, dtype: int64
expert_eligibility  excluded  included  not applicable  not enough information  not excluded  not included
criterion_type                                                                                            
exclusion                 10         0              46                     119           483             0
inclusion                  0       15

In [18]:
%cd /kaggle/working/clinical_trial_matching_llm
!ls -la outputs
!ls -la outputs/kv

/kaggle/working/clinical_trial_matching_llm
total 20
drwxr-xr-x 5 root root 4096 Nov  8 16:35 .
drwxr-xr-x 7 root root 4096 Nov  8 16:56 ..
drwxr-xr-x 2 root root 4096 Nov  8 16:30 eval
drwxr-xr-x 2 root root 4096 Nov  8 16:48 kv
drwxr-xr-x 3 root root 4096 Nov  8 16:30 lora_r16
total 116
drwxr-xr-x 2 root root  4096 Nov  8 16:48 .
drwxr-xr-x 5 root root  4096 Nov  8 16:35 ..
-rw-r--r-- 1 root root  4282 Nov  8 16:48 benchmark.json
-rw-r--r-- 1 root root 10174 Nov  8 16:55 correctness_fp16.json
-rw-r--r-- 1 root root 10991 Nov  8 16:43 correctness_fp32.json
-rw-r--r-- 1 root root 74079 Nov  8 16:48 kv_benchmark.png


In [20]:
%cd /kaggle/working/clinical_trial_matching_llm
!zip -r -0 kv_results.zip outputs/kv
!mv kv_results.zip /kaggle/working/
!ls -la /kaggle/working/kv_results.zip

/kaggle/working/clinical_trial_matching_llm
  adding: outputs/kv/ (stored 0%)
  adding: outputs/kv/benchmark.json (stored 0%)
  adding: outputs/kv/kv_benchmark.png (stored 0%)
  adding: outputs/kv/correctness_fp16.json (stored 0%)
  adding: outputs/kv/correctness_fp32.json (stored 0%)
-rw-r--r-- 1 root root 100442 Nov  8 16:58 /kaggle/working/kv_results.zip


In [21]:
import json
for d in ("fp16", "fp32"):
    r = json.load(open(f"outputs/kv/correctness_{d}.json"))
    print(f"\n=== correctness {d} ===")
    for x in r["synthetic"]:
        print("synthetic", x["input_len"], x["new_tokens"], "equal:", x["equal"],
              "max_logit_diff: %.4f" % x["max_abs_logit_diff"], "argmax_agree: %.3f" % x["argmax_agreement"])
    eq = sum(x["equal"] for x in r["real"])
    print(f"real prompts identical: {eq}/{len(r['real'])}")
    for x in r["real"]:
        if not x["equal"]:
            print("  MISMATCH id", x["annotation_id"], "first_mismatch", x["first_mismatch"],
                  "top2_gap_at_mismatch", x.get("top2_gap_at_first_mismatch"))

b = json.load(open("outputs/kv/benchmark.json"))
print("\n=== benchmark ===", b["gpu"])
for x in b["rows"]:
    print(f"in={x['input_len']:5d} out={x['new_tokens']:4d} | uncached {x['uncached']['total_tps']:7.1f} tok/s"
          f" | cached {x['cached']['total_tps']:7.1f} tok/s | speedup {x['speedup']:.1f}x"
          f" | peak MiB u/c {x['uncached']['peak_mib']:.0f}/{x['cached']['peak_mib']:.0f}"
          f" | prefill ms {x['cached']['prefill_ms']:.0f} | decode ms {x['cached']['decode_ms']:.0f}"
          f" | equal={x['tokens_equal']}")


=== correctness fp16 ===
synthetic 64 32 equal: True max_logit_diff: 0.0557 argmax_agree: 1.000
synthetic 256 64 equal: True max_logit_diff: 0.0547 argmax_agree: 1.000
synthetic 512 128 equal: True max_logit_diff: 0.0547 argmax_agree: 1.000
synthetic 1024 128 equal: True max_logit_diff: 0.0400 argmax_agree: 1.000
real prompts identical: 30/30

=== correctness fp32 ===
synthetic 64 32 equal: True max_logit_diff: 0.0001 argmax_agree: 1.000
synthetic 256 64 equal: True max_logit_diff: 0.0002 argmax_agree: 1.000
synthetic 512 128 equal: True max_logit_diff: 0.0002 argmax_agree: 1.000
synthetic 1024 128 equal: True max_logit_diff: 0.0001 argmax_agree: 1.000
real prompts identical: 30/30

=== benchmark === Tesla T4
in=  128 out=  32 | uncached    22.9 tok/s | cached    25.7 tok/s | speedup 1.1x | peak MiB u/c 10/12 | prefill ms 46 | decode ms 1200 | equal=True
in=  128 out= 128 | uncached    20.1 tok/s | cached    25.8 tok/s | speedup 1.3x | peak MiB u/c 18/13 | prefill ms 47 | decode ms 49

In [23]:
%cd /kaggle/working/clinical_trial_matching_llm
!rm -f data/processed/test.jsonl data/processed/train.jsonl data/processed/val.jsonl
!rm -f outputs/kv/benchmark.json outputs/kv/correctness_fp16.json outputs/kv/correctness_fp32.json outputs/kv/kv_benchmark.png
!git pull
!ls data/processed outputs/kv src
!ls outputs/lora_r16/epoch_3

/kaggle/working/clinical_trial_matching_llm
Updating b2eb37c..cff4176
Fast-forward
 data/processed/test.jsonl        | 206 ++++++++++++++
 data/processed/train.jsonl       | 602 +++++++++++++++++++++++++++++++++++++++
 data/processed/val.jsonl         | 206 ++++++++++++++
 outputs/kv/benchmark.json        | 145 ++++++++++
 outputs/kv/correctness_fp16.json | 381 +++++++++++++++++++++++++
 outputs/kv/correctness_fp32.json | 381 +++++++++++++++++++++++++
 outputs/kv/kv_benchmark.png      | Bin 0 -> 74079 bytes
 src/attention.py                 |  93 ++++++
 src/test_attention.py            | 152 ++++++++++
 9 files changed, 2166 insertions(+)
 create mode 100644 data/processed/test.jsonl
 create mode 100644 data/processed/train.jsonl
 create mode 100644 data/processed/val.jsonl
 create mode 100644 outputs/kv/benchmark.json
 create mode 100644 outputs/kv/correctness_fp16.json
 create mode 100644 outputs/kv/correctness_fp32.json
 create mode 100644 outputs/kv/kv_benchmark.png
 create mode 1

In [24]:
!python -m src.test_attention


--- hand-checkable example (L=4, D=2, float64) ---
reference:
 [[1.         2.        ]
 [2.3395231  3.3395231 ]
 [3.51046953 4.51046953]
 [4.06544694 5.06544694]] 
tiled:
 [[1.         2.        ]
 [2.3395231  3.3395231 ]
 [3.51046953 4.51046953]
 [4.06544694 5.06544694]]
OK: row 0 == v[0]; changing the LAST token does not change earlier outputs (causality)

--- numerical stability demo (fp32, huge scores) ---
{'naive_has_nan_or_inf': True, 'tiled_has_nan_or_inf': False, 'tiled_max_err_vs_fp64': 7.331365213580909e-06, 'max_score': 211.04820251464844}

--- correctness vs float64 gold (max / mean abs error) ---
tiny                                   float64  tiled max=1.11e-16 mean=2.30e-17 | ref max=0.00e+00 | tiled-vs-ref=1.11e-16 | PASS
tiny                                   float32  tiled max=1.44e-08 mean=5.81e-09 | ref max=3.16e-08 | tiled-vs-ref=3.73e-08 | PASS
tiny                                   float16  tiled max=9.88e-05 mean=2.70e-05 | ref max=9.88e-05 | tiled-vs-ref=0.00

In [25]:
%cd /kaggle/working/clinical_trial_matching_llm
!zip -r -0 attention_results.zip outputs/attention
!mv attention_results.zip /kaggle/working/

/kaggle/working/clinical_trial_matching_llm
  adding: outputs/attention/ (stored 0%)
  adding: outputs/attention/attention_results.json (stored 0%)
  adding: outputs/attention/attention_benchmark.png (stored 0%)


In [26]:
%cd /kaggle/working/clinical_trial_matching_llm
!git pull
!ls outputs/lora_r16/epoch_3

/kaggle/working/clinical_trial_matching_llm
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 10 (delta 3), reused 10 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 125.14 KiB | 2.12 MiB/s, done.
From https://github.com/Maryam-Faatima/clinical_trial_matching_llm
   cff4176..e41cb0f  main       -> origin/main
Updating cff4176..e41cb0f
error: The following untracked working tree files would be overwritten by merge:
	outputs/attention/attention_benchmark.png
	outputs/attention/attention_results.json
Please move or remove them before you merge.
Aborting
adapter_config.json  adapter_model.safetensors	README.md


In [29]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!BASE_MODEL=Qwen/Qwen2.5-0.5B-Instruct OUT_DIR=outputs/lora_draft python -m src.train_lora

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 242.48it/s]
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
epoch 1 step 10/114 loss 0.1036
epoch 1 step 20/114 loss 0.1670
epoch 1 step 30/114 loss 0.1544
== epoch 1 val_loss 0.1327  (282s)
epoch 2 step 40/114 loss 0.0953
epoch 2 step 50/114 loss 0.0661
epoch 2 step 60/114 loss 0.1034
epoch 2 step 70/114 loss 0.1115
== epoch 2 val_loss 0.1255  (565s)
epoch 3 step 80/114 loss 0.0688
epoch 3 step 90/114 loss 0.0796
epoch 3 step 100/114 loss 0.0252
epoch 3 step 110/114 loss 0.0729
== epoch 3 val_loss 0.1300  (848s)
peak GPU mem GiB: 7.94559383392334


In [39]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
%env BASE_MODEL=Qwen/Qwen2.5-0.5B-Instruct
%env OUT_DIR=outputs/lora_draft
!python -m src.train_lora

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
env: BASE_MODEL=Qwen/Qwen2.5-0.5B-Instruct
env: OUT_DIR=outputs/lora_draft
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 3.07MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 22.0MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 48.2MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 123MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 151MB/s]

model.safetensors: downloading bytes: ▌                     | 22.9MB            
model.safetensors: downloading bytes: ██▏                   | 99.0MB, 6.61MB/s  
model.safetensors: downloading bytes: ███                   |  135MB, 9.68MB/s  
model.safetensors: downloading bytes: ██████████████████▋   |  840MB, 65.8MB/s  
model.safetensors: downloading bytes: ██████████████████████|  854MB, 63.7MB/s  
model.safetensors: reconstructing file: 100%|██████|  988MB 

In [46]:
!python -m src.bench_spec --n 5 --ks 3 --repeats 1 --name smoke

Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 249.25it/s]
[A_real] baseline cached greedy k=0 | equal 5/5 |   24.2 tok/s |     587 ms/prompt | speedup 1.00x | peak 2984 MiB
[B_long] baseline cached greedy k=0 | equal 2/2 |   25.3 tok/s |    3799 ms/prompt | speedup 1.00x | peak 3066 MiB
[A_real] self (oracle)          k=3 | equal 5/5 |   20.3 tok/s |     698 ms/prompt | speedup 0.84x | peak 2993 MiB | accept 1.00 | tok/round 3.30
[B_long] self (oracle)          k=3 | equal 2/2 |   25.2 tok/s |    3813 ms/prompt | speedup 1.00x | peak 3088 MiB | accept 1.00 | tok/round 3.96
Loading weights: 100%|███████████████████████| 290/290 [00:00<00:00, 717.10it/s]
[A_real] base 0.5B              k=3 | equal 5/5 |   19.7 tok/s |     722 ms/prompt | speedup 0.81x | peak 3927 MiB | accept 0.72 | tok/round 2.64
[B_long] base 0.5B              k=3 | equal 2/2 |   19.8 tok/s |    4856 ms/prompt | speedup 0.78x | peak 4016 MiB | accept 0.57 | tok/round 2.68
Loading weights: 100%|██

In [47]:
!python -m src.bench_spec --hf-ref

Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 245.34it/s]
[A_real] baseline cached greedy k=0 | equal 40/40 |   24.9 tok/s |     610 ms/prompt | speedup 1.00x | peak 2990 MiB
[B_long] baseline cached greedy k=0 | equal 2/2 |   25.1 tok/s |    3832 ms/prompt | speedup 1.00x | peak 3067 MiB
[A_real] self (oracle)          k=2 | equal 40/40 |   21.8 tok/s |     696 ms/prompt | speedup 0.88x | peak 3000 MiB | accept 1.00 | tok/round 2.70
[A_real] self (oracle)          k=3 | equal 40/40 |   20.6 tok/s |     737 ms/prompt | speedup 0.83x | peak 3000 MiB | accept 1.00 | tok/round 3.40
[A_real] self (oracle)          k=4 | equal 40/40 |   21.5 tok/s |     706 ms/prompt | speedup 0.86x | peak 3000 MiB | accept 1.00 | tok/round 4.46
[A_real] self (oracle)          k=6 | equal 40/40 |   21.6 tok/s |     702 ms/prompt | speedup 0.87x | peak 3000 MiB | accept 1.00 | tok/round 6.37
[B_long] self (oracle)          k=2 | equal 2/2 |   25.1 tok/s |    3831 ms/prompt | speedup 1.

In [48]:
!python -m src.bench_spec --dtype fp32 --n 20 --ks 4 --repeats 1 --name spec_fp32

Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 213.41it/s]
[A_real] baseline cached greedy k=0 | equal 20/20 |   18.2 tok/s |     795 ms/prompt | speedup 1.00x | peak 5954 MiB
[B_long] baseline cached greedy k=0 | equal 2/2 |   24.4 tok/s |    3934 ms/prompt | speedup 1.00x | peak 6037 MiB
[A_real] self (oracle)          k=4 | equal 20/20 |   11.5 tok/s |    1260 ms/prompt | speedup 0.63x | peak 5972 MiB | accept 1.00 | tok/round 4.43
[B_long] self (oracle)          k=4 | equal 2/2 |   19.4 tok/s |    4957 ms/prompt | speedup 0.79x | peak 6080 MiB | accept 1.00 | tok/round 5.00
Loading weights: 100%|███████████████████████| 290/290 [00:00<00:00, 583.06it/s]
[A_real] base 0.5B              k=4 | equal 20/20 |   14.6 tok/s |     990 ms/prompt | speedup 0.80x | peak 7839 MiB | accept 0.84 | tok/round 3.70
[B_long] base 0.5B              k=4 | equal 2/2 |   15.5 tok/s |    6198 ms/prompt | speedup 0.63x | peak 7935 MiB | accept 0.48 | tok/round 2.88
Loading weights: 1

In [49]:
!zip -r -0 spec_results.zip outputs/speculative outputs/lora_draft/train_log.json outputs/lora_draft/training_curves.png
!mv spec_results.zip /kaggle/working/

  adding: outputs/speculative/ (stored 0%)
  adding: outputs/speculative/smoke.png (stored 0%)
  adding: outputs/speculative/spec.json (stored 0%)
  adding: outputs/speculative/smoke.json (stored 0%)
  adding: outputs/speculative/spec.png (stored 0%)
  adding: outputs/speculative/spec_fp32.png (stored 0%)
  adding: outputs/speculative/spec_fp32.json (stored 0%)
  adding: outputs/lora_draft/train_log.json (stored 0%)
  adding: outputs/lora_draft/training_curves.png (stored 0%)


In [34]:
%cd /kaggle/working/clinical_trial_matching_llm
!grep -n "BASE_MODEL\|OUT_DIR" src/train_lora.py
!ls outputs

/kaggle/working/clinical_trial_matching_llm
attention  eval  kv  lora_r16


In [36]:
%cd /kaggle/working/clinical_trial_matching_llm
import glob, os, shutil
!wget -q -O /tmp/adapter_epoch_3.zip https://github.com/Maryam-Faatima/clinical_trial_matching_llm/releases/download/v1.0/adapter_epoch_3.zip
!rm -rf /tmp/ad && mkdir -p /tmp/ad
!unzip -q -o /tmp/adapter_epoch_3.zip -d /tmp/ad
src = os.path.dirname(glob.glob("/tmp/ad/**/adapter_config.json", recursive=True)[0])
for f in os.listdir(src):
    shutil.copy(os.path.join(src, f), "outputs/lora_r16/epoch_3/")

/kaggle/working/clinical_trial_matching_llm


In [37]:
!git pull
!grep -n "BASE_MODEL\|OUT_DIR" src/train_lora.py
!ls src

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 344 bytes | 344.00 KiB/s, done.
From https://github.com/Maryam-Faatima/clinical_trial_matching_llm
   e41cb0f..f200cad  main       -> origin/main
Updating cff4176..f200cad
error: The following untracked working tree files would be overwritten by merge:
	outputs/attention/attention_benchmark.png
	outputs/attention/attention_results.json
Please move or remove them before you merge.
Aborting
attention.py  evaluate.py   metrics.py	   __pycache__	      token_stats.py
bench_kv.py   __init__.py   model_info.py  test_attention.py  train_lora.py
data_prep.py  kv_decode.py  prompts.py	   test_kv.py


In [38]:
%cd /kaggle/working/clinical_trial_matching_llm
!rm -f outputs/attention/attention_benchmark.png outputs/attention/attention_results.json
!git pull
!grep -n "BASE_MODEL\|OUT_DIR" src/train_lora.py
!ls src

/kaggle/working/clinical_trial_matching_llm
Updating cff4176..f200cad
Fast-forward
 outputs/attention/attention_benchmark.png | Bin 0 -> 129982 bytes
 outputs/attention/attention_results.json  | 661 ++++++++++++++++++++++++++++++
 src/bench_spec.py                         | 194 +++++++++
 src/speculative.py                        | 104 +++++
 src/train_lora.py                         |   6 +-
 5 files changed, 963 insertions(+), 2 deletions(-)
 create mode 100644 outputs/attention/attention_benchmark.png
 create mode 100644 outputs/attention/attention_results.json
 create mode 100644 src/bench_spec.py
 create mode 100644 src/speculative.py
21:MODEL = os.environ.get("BASE_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
22:OUT = Path(os.environ.get("OUT_DIR", "outputs/lora_r16"))
attention.py   evaluate.py   model_info.py   test_attention.py
bench_kv.py    __init__.py   prompts.py      test_kv.py
bench_spec.py  kv_decode.py  __pycache__     token_stats.py
data_prep.py   metrics.py    speculative.p

In [44]:
%cd /kaggle/working/clinical_trial_matching_llm
!git pull
!grep -n "crop_cache" src/speculative.py

/kaggle/working/clinical_trial_matching_llm
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 635 bytes | 635.00 KiB/s, done.
From https://github.com/Maryam-Faatima/clinical_trial_matching_llm
   f200cad..a1bbb3d  main       -> origin/main
Updating f200cad..a1bbb3d
Fast-forward
 src/speculative.py | 18 ++++++++++++++++--
 1 file changed, 16 insertions(+), 2 deletions(-)
23:def crop_cache(cache, new_len):
110:        crop_cache(t_cache, len(committed) - 1)
112:        crop_cache(d_cache, valid_d); d_len = valid_d


In [1]:
%cd /kaggle/working/clinical_trial_matching_llm
import json, glob
for f in sorted(glob.glob("outputs/speculative/*.json")):
    r = json.load(open(f))
    print(f"\n=== {f} | {r['gpu']} | {r['dtype']} | repeats={r['repeats']} ===")
    for x in r["rows"]:
        acc = f"accept {x['acceptance_rate']:.2f} tok/round {x['tokens_per_round']:.2f}" if "acceptance_rate" in x else ""
        print(f"[{x['workload']}] {x['draft'][:24]:24s} k={x['k']} | equal {x['n_equal']}/{x['n_prompts']}"
              f" | {x['tok_per_s']:5.1f} tok/s | {x['ms_per_prompt']:6.0f} ms | speedup {x['speedup']:.2f}x"
              f" | peak {x['peak_mib']:.0f} MiB {acc}")
        if x.get("mismatches"):
            print("   MISMATCH:", x["mismatches"])

[Errno 2] No such file or directory: '/kaggle/working/clinical_trial_matching_llm'
/kaggle/working


In [2]:
d = json.load(open("outputs/lora_draft/train_log.json"))
print("trainable:", d["trainable_params"], "| peak GiB:", round(d["peak_gpu_mem_gib"], 2), "| seconds:", round(d["train_seconds"]))
print([(e["epoch"], round(e["val_loss"], 4)) for e in d["epochs"]])

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/lora_draft/train_log.json'

In [3]:
%cd /kaggle/working
import os
if not os.path.exists("clinical_trial_matching_llm"):
    !git clone https://github.com/Maryam-Faatima/clinical_trial_matching_llm.git
%cd /kaggle/working/clinical_trial_matching_llm
!git pull
!pip install -q matplotlib
!python -m src.plot_spec_from_log

/kaggle/working
Cloning into 'clinical_trial_matching_llm'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 105 (delta 38), reused 93 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 368.06 KiB | 3.57 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/kaggle/working/clinical_trial_matching_llm
Already up to date.
token-equality: 44/44 configurations fully identical to cached greedy

full fp16 run (A_real: n=40 validation prompts)
draft                       k  accept tok/round  tok/s  speedup
baseline cached greedy      0       -         -   24.9     1.00
self (oracle)               2    1.00      2.70   21.8     0.88
self (oracle)               3    1.00      3.40   20.6     0.83
self (oracle)               4    1.00      4.46   21.5     0.86
self (oracle)               6    1.00      6.37   21.6     0.87
base 0.5B                   2    0.84      2.38   22.7 

In [4]:
import glob, shutil
!mkdir -p outputs/lora_r16/epoch_3
!wget -q -O /tmp/adapter_epoch_3.zip https://github.com/Maryam-Faatima/clinical_trial_matching_llm/releases/download/v1.0/adapter_epoch_3.zip
!rm -rf /tmp/ad && mkdir -p /tmp/ad
!unzip -q -o /tmp/adapter_epoch_3.zip -d /tmp/ad
src = os.path.dirname(glob.glob("/tmp/ad/**/adapter_config.json", recursive=True)[0])
for f in os.listdir(src):
    shutil.copy(os.path.join(src, f), "outputs/lora_r16/epoch_3/")

!pip uninstall -y torchao
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
%env BASE_MODEL=Qwen/Qwen2.5-0.5B-Instruct
%env OUT_DIR=outputs/lora_draft
!python -m src.train_lora

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
env: BASE_MODEL=Qwen/Qwen2.5-0.5B-Instruct
env: OUT_DIR=outputs/lora_draft
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 2.50MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 16.5MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 46.6MB/s]
merges.txt: 100%|██████████████████████████| 1.67M/1.67M [00:00<00:00, 95.6MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 135MB/s]

model.safetensors: downloading bytes: ▏                     | 9.26MB            
model.safetensors: downloading bytes: ██████▋               |  298MB, 22.9MB/s  
model.safetensors: downloading bytes: ███████████▎          |  509MB, 42.2MB/s  
model.safetensors: downloading bytes: ███████████████████   |  854MB, 75.1MB/s  
model.safetensors: reconst

In [5]:
!zip -r -0 draft_results.zip outputs/speculative outputs/lora_draft
!mv draft_results.zip /kaggle/working/

  adding: outputs/speculative/ (stored 0%)
  adding: outputs/speculative/spec_from_log.png (stored 0%)
  adding: outputs/speculative/spec_from_log.json (stored 0%)
  adding: outputs/lora_draft/ (stored 0%)
  adding: outputs/lora_draft/epoch_3/ (stored 0%)
  adding: outputs/lora_draft/epoch_3/adapter_model.safetensors (stored 0%)
  adding: outputs/lora_draft/epoch_3/README.md (stored 0%)
  adding: outputs/lora_draft/epoch_3/adapter_config.json (stored 0%)
  adding: outputs/lora_draft/tokenizer_config.json (stored 0%)
  adding: outputs/lora_draft/train_log.json (stored 0%)
  adding: outputs/lora_draft/tokenizer.json (stored 0%)
  adding: outputs/lora_draft/epoch_1/ (stored 0%)
  adding: outputs/lora_draft/epoch_1/adapter_model.safetensors (stored 0%)
  adding: outputs/lora_draft/epoch_1/README.md (stored 0%)
  adding: outputs/lora_draft/epoch_1/adapter_config.json (stored 0%)
  adding: outputs/lora_draft/training_curves.png (stored 0%)
  adding: outputs/lora_draft/epoch_2/ (stored 0%)
  

In [6]:
!python -m src.evaluate --split test --name base_test --allow-test
!python -m src.evaluate --split test --adapter outputs/lora_r16/epoch_3 --name ft_e3_test --allow-test

config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 1.67MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 15.1MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 40.3MB/s]
merges.txt: 100%|██████████████████████████| 1.67M/1.67M [00:00<00:00, 99.0MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 125MB/s]

model.safetensors: downloading bytes:                       |  0.00B            
model.safetensors: downloading bytes: ▍                     | 55.4MB, 3.18MB/s  
model.safetensors: downloading bytes: ▉                     |  136MB, 10.6MB/s  
model.safetensors: downloading bytes: █▏                    |  169MB, 12.9MB/s  
model.safetensors: downloading bytes: █▋                    |  243MB, 18.8MB/s  
model.safetensors: downloading bytes: █▉                    |  280MB, 23.8MB/s  
model.safetensors: downloading bytes: ██▏                   |  309MB, 25.6MB/s  
model.safetensors: download

In [8]:
%cd /kaggle/working/clinical_trial_matching_llm
!zip -r -0 test_eval.zip outputs/eval
!mv test_eval.zip /kaggle/working/

/kaggle/working/clinical_trial_matching_llm
  adding: outputs/eval/ (stored 0%)
  adding: outputs/eval/ft_e2_val_metrics.json (stored 0%)
  adding: outputs/eval/ft_e1_val_predictions.jsonl (stored 0%)
  adding: outputs/eval/base_test_predictions.jsonl (stored 0%)
  adding: outputs/eval/base_val_metrics.json (stored 0%)
  adding: outputs/eval/ft_e3_test_predictions.jsonl (stored 0%)
  adding: outputs/eval/ft_e3_test_metrics.json (stored 0%)
  adding: outputs/eval/ft_e1_val_metrics.json (stored 0%)
  adding: outputs/eval/base_val_predictions.jsonl (stored 0%)
  adding: outputs/eval/ft_e3_val_metrics.json (stored 0%)
  adding: outputs/eval/ft_e3_val_predictions.jsonl (stored 0%)
  adding: outputs/eval/base_test_metrics.json (stored 0%)
  adding: outputs/eval/ft_e2_val_predictions.jsonl (stored 0%)


In [3]:
%cd /kaggle/working/clinical_trial_matching_llm
!git pull
!python -c "import gradio; print('gradio', gradio.__version__)"
!python -m src.selftest_app

/kaggle/working/clinical_trial_matching_llm
Already up to date.
gradio 6.30.0
Adapter not found in outputs/lora_r16/epoch_3; downloading from https://github.com/Maryam-Faatima/clinical_trial_matching_llm/releases/download/v1.0/adapter_epoch_3.zip ...
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 3.86MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 30.5MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 45.8MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 128MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 165MB/s]

model.safetensors: downloading bytes:                       |  0.00B            
model.safetensors: downloading bytes: ▍                     | 60.9MB, 3.56MB/s  
model.safetensors: downloading bytes: █▏                    |  173MB, 11.8MB/s  
model.safetensors: downloading bytes: █▌                    |  211MB, 18.2MB/s  
model.safetensors: 

In [4]:
%cd /kaggle/working/clinical_trial_matching_llm
!pip uninstall -y torchao
!python -m src.selftest_app

/kaggle/working/clinical_trial_matching_llm
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 246.54it/s]
device: cuda | dtype: torch.float16 | adapter: local files
criterion: inclusion - Patients of chronic stable angina with abnormal Exercise Myocardial Perfusion Sp | gold: not enough information
[KV cache (default)] label=not enough information evidence=[] status=unresolved time=2.61s raw='{"label": "not enough information", "evidence": []}'
[No cache (reference)] label=not enough information evidence=[] status=unresolved time=1.03s raw='{"label": "not enough information", "evidence": []}'
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 2.25MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 14.4MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 44.6MB/s]
merges.txt: 100%|██████

In [5]:
!python app.py --share

Loading model (first start downloads the base model, about 3 GB) ...
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 244.52it/s]
Model ready on cuda. Starting the web app ...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://948026f9c3faf88890.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Loading weights: 100%|███████████████████████| 290/290 [00:00<00:00, 580.45it/s]
^C
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://948026f9c3faf88890.gradio.live
